# 小鼠 pupil 提取与直径计算演示

这个 notebook 将原始图像处理流程拆成可逐步运行的 cell，用于演示如何自动定位眼睛 ROI、提取 pupil 边界、拟合圆并计算直径。

## 1. 导入依赖与显示工具

In [ ]:
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

plt.rcParams["figure.dpi"] = 120


def show_img(title, img, cmap="gray", figsize=(6, 5)):
    plt.figure(figsize=figsize)
    plt.imshow(img, cmap=cmap)
    plt.title(title)
    plt.axis("off")
    plt.show()


def show_overlay(title, gray, overlay, figsize=(7, 5)):
    plt.figure(figsize=figsize)
    plt.imshow(gray, cmap="gray")
    plt.imshow(overlay, alpha=0.45)
    plt.title(title)
    plt.axis("off")
    plt.show()


## 2. 设置图片路径和可调参数

把 `img_path` 改成要分析的小鼠眼部图片路径。`micron_per_px` 是可选校准参数；如果还没有像素到真实长度的换算，先保持为 `None`。

In [ ]:
img_path = Path("pupiltest.jpeg")

# 如果已经知道图像比例尺，例如 1 px = 3.2 um，就填 3.2；否则保持 None。
micron_per_px = None

# 是否显示每一步中间结果。
debug = True


## 3. 读取并检查原始图像

In [ ]:
gray = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)

if gray is None:
    raise FileNotFoundError(f"Cannot read image: {img_path.resolve()}")

print(f"Image path: {img_path}")
print(f"Image shape: {gray.shape[1]} x {gray.shape[0]} px")
print(f"Intensity range: {gray.min()} - {gray.max()}")
show_img("Original gray image", gray, figsize=(7, 5))

## 4. 几何工具函数

`fit_circle_least_squares` 用轮廓点做最小二乘圆拟合，`expand_bbox` 把第一轮候选区域扩大为眼睛 ROI。

In [ ]:
def odd_kernel_size(value, minimum=3):
    """Return an odd kernel size suitable for OpenCV morphology/blur."""
    value = int(round(value))
    value = max(minimum, value)
    if value % 2 == 0:
        value += 1
    return value


def full_image_scale_params(gray_shape):
    """Scale full-image search thresholds with image size."""
    H, W = gray_shape
    image_area = H * W
    short_side = min(H, W)
    return {
        "full_blur_sigma": max(2.0, short_side * 0.0064),
        "full_blackhat_kernel": odd_kernel_size(short_side * 0.082, minimum=31),
        "seed_area_min": max(20, int(image_area * 0.00012)),
        "seed_area_max": max(80, int(image_area * 0.0028)),
        "fallback_area_min": max(80, int(image_area * 0.0006)),
        "fallback_area_max": max(600, int(image_area * 0.013)),
        "seed_min_roi_w": max(45, int(short_side * 0.11)),
        "seed_min_roi_h": max(40, int(short_side * 0.095)),
    }


def roi_scale_params(roi_shape):
    """Scale ROI-level pupil extraction thresholds with ROI size."""
    h, w = roi_shape
    roi_area = h * w
    short_side = min(h, w)
    return {
        "roi_blur_sigma": max(1.0, short_side * 0.025),
        "pupil_blackhat_kernel": odd_kernel_size(short_side * 0.32, minimum=15),
        "pupil_area_min": max(8, int(roi_area * 0.003)),
        "pupil_area_max": max(80, int(roi_area * 0.14)),
        "glint_threshold": None,
    }


def fit_circle_least_squares(points):
    """
    用最小二乘拟合圆:
    (x - cx)^2 + (y - cy)^2 = r^2
    """
    points = points.reshape(-1, 2).astype(np.float64)
    x = points[:, 0]
    y = points[:, 1]

    A = np.column_stack([2 * x, 2 * y, np.ones_like(x)])
    b = x ** 2 + y ** 2

    c, _, _, _ = np.linalg.lstsq(A, b, rcond=None)
    cx, cy, d = c

    r = np.sqrt(cx ** 2 + cy ** 2 + d)
    return float(cx), float(cy), float(r)


def circle_fit_residual(points, cx, cy, r):
    """Mean absolute radial residual of a fitted circle, in pixels."""
    pts = points.reshape(-1, 2).astype(np.float64)
    radial = np.sqrt((pts[:, 0] - cx) ** 2 + (pts[:, 1] - cy) ** 2)
    return float(np.mean(np.abs(radial - r)))


def expand_bbox(x, y, w, h, img_shape, scale_x=2.3, scale_y=2.0):
    """
    扩大 bbox，得到眼睛 ROI。
    对这类小鼠眼图，向左和向上多扩一点，因为 pupil 常在反光点左上方。
    """
    H, W = img_shape
    cx = x + w / 2
    cy = y + h / 2

    new_w = int(w * scale_x)
    new_h = int(h * scale_y)
    new_x = int(cx - new_w * 0.58)
    new_y = int(cy - new_h * 0.55)

    new_x = max(0, new_x)
    new_y = max(0, new_y)
    new_w = min(new_w, W - new_x)
    new_h = min(new_h, H - new_y)

    return new_x, new_y, new_w, new_h


def estimate_eye_fissure_features(roi_gray):
    """
    Estimate whether an ROI contains a horizontal dark elliptical eye fissure.
    Returns a compact score and interpretable measurements for candidate ranking/QC.
    """
    h, w = roi_gray.shape
    if h < 10 or w < 10:
        return {
            "fissure_score": 0.0,
            "fissure_area_ratio": 0.0,
            "fissure_aspect": 0.0,
            "fissure_horizontal": 0.0,
        }

    blurred = cv2.GaussianBlur(roi_gray, (0, 0), sigmaX=max(1.0, min(h, w) * 0.025))
    _, dark = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 3))
    dark = cv2.morphologyEx(dark, cv2.MORPH_CLOSE, k, iterations=1)
    dark = cv2.morphologyEx(dark, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3)))

    contours, _ = cv2.findContours(dark, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    best = None
    roi_area = h * w

    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area < roi_area * 0.015 or area > roi_area * 0.75:
            continue
        x, y, bw, bh = cv2.boundingRect(cnt)
        bbox_aspect = bw / max(1, bh)
        area_ratio = area / roi_area
        horizontal = min(1.0, bbox_aspect / 2.2)
        # Eye fissure should be horizontally elongated, but not necessarily a perfect ellipse.
        score = min(area_ratio / 0.28, 1.0) * min(bbox_aspect / 2.2, 1.4) * horizontal
        item = {
            "fissure_score": float(score),
            "fissure_area_ratio": float(area_ratio),
            "fissure_aspect": float(bbox_aspect),
            "fissure_horizontal": float(horizontal),
        }
        if best is None or item["fissure_score"] > best["fissure_score"]:
            best = item

    if best is None:
        return {
            "fissure_score": 0.0,
            "fissure_area_ratio": 0.0,
            "fissure_aspect": 0.0,
            "fissure_horizontal": 0.0,
        }
    return best


def candidate_table(candidates):
    """Convert candidate dictionaries into a display-friendly numeric table."""
    rows = []
    for idx, cand in enumerate(candidates):
        row = {"candidate_id": idx}
        for key, value in cand.items():
            if key == "contour":
                continue
            if isinstance(value, tuple):
                for j, item in enumerate(value):
                    row[f"{key}_{j}"] = item
            elif isinstance(value, (int, float, np.integer, np.floating, str)):
                row[key] = value
        rows.append(row)
    return pd.DataFrame(rows)


def simple_kmeans(X, n_clusters=3, max_iter=100, random_state=0):
    """Small deterministic KMeans implementation to avoid optional sklearn runtime issues."""
    rng = np.random.default_rng(random_state)
    if len(X) <= n_clusters:
        return np.arange(len(X))

    # Initialize centers from well-spaced points along the first principal dimension.
    order = np.argsort(X[:, 0])
    init_idx = np.linspace(0, len(order) - 1, n_clusters).round().astype(int)
    centers = X[order[init_idx]].copy()

    labels = np.zeros(len(X), dtype=int)
    for _ in range(max_iter):
        distances = np.linalg.norm(X[:, None, :] - centers[None, :, :], axis=2)
        new_labels = np.argmin(distances, axis=1)
        new_centers = centers.copy()
        for k in range(n_clusters):
            members = X[new_labels == k]
            if len(members) > 0:
                new_centers[k] = members.mean(axis=0)
            else:
                new_centers[k] = X[rng.integers(0, len(X))]
        if np.array_equal(labels, new_labels):
            break
        labels = new_labels
        centers = new_centers
    return labels



def pc_formula_text(loading_df, explained_ratio, top_n=3):
    """Create compact PC loading formulas for plot annotation."""
    aliases = {
        "score": "score",
        "area": "area",
        "circularity": "circ",
        "dark_score": "dark",
        "bright_ratio": "bright",
        "mean_seed_intensity": "seed_int",
        "fissure_score": "fis_score",
        "fissure_area_ratio": "fis_area",
        "fissure_aspect": "fis_aspect",
        "fissure_horizontal": "fis_horiz",
    }

    def format_terms(pc_name):
        loading_col = f"{pc_name}_loading"
        abs_col = f"abs_{pc_name}_loading"
        top = loading_df.sort_values(abs_col, ascending=False).head(top_n)
        terms = []
        for _, row in top.iterrows():
            coef = float(row[loading_col])
            sign = "+" if coef >= 0 else "-"
            feature = aliases.get(row["feature"], row["feature"])
            terms.append(f" {sign} {abs(coef):.2f}z({feature})")
        formula = f"{pc_name} =" + "".join(terms)
        return formula.replace("= +", "=")

    pc1_pct = 100 * float(explained_ratio[0])
    pc2_pct = 100 * float(explained_ratio[1])
    return "\n".join([
        f"PC composition, top {top_n} loadings",
        f"{format_terms('PC1')}  [{pc1_pct:.1f}%]",
        f"{format_terms('PC2')}  [{pc2_pct:.1f}%]",
    ])


def plot_candidate_clusters(candidates, selected_roi=None, title="ROI candidate clusters", save_path=None):
    """
    Plot ROI candidate feature clusters using standardized features, SVD/PCA, and KMeans-like clustering.
    The final selected ROI is highlighted with a star.
    """
    df = candidate_table(candidates)
    if df.empty:
        print("No candidates to plot.")
        return df

    feature_cols = [
        c for c in [
            "score", "area", "circularity", "dark_score", "bright_ratio",
            "mean_seed_intensity", "fissure_score", "fissure_area_ratio",
            "fissure_aspect", "fissure_horizontal",
        ] if c in df.columns
    ]
    numeric = df[feature_cols].apply(pd.to_numeric, errors="coerce").fillna(0.0)
    if len(df) < 2 or len(feature_cols) < 2:
        display(df)
        return df

    X_raw = numeric.to_numpy(dtype=float)
    feature_mean = X_raw.mean(axis=0)
    feature_std = np.where(X_raw.std(axis=0) == 0, 1, X_raw.std(axis=0))
    X = (X_raw - feature_mean) / feature_std
    _, singular_values, vt = np.linalg.svd(X, full_matrices=False)
    xy = X @ vt[:2].T
    explained_variance = singular_values ** 2 / max(1, len(X) - 1)
    explained_ratio = explained_variance / explained_variance.sum()

    loading_df = pd.DataFrame({
        "feature": feature_cols,
        "PC1_loading": vt[0, :],
        "PC2_loading": vt[1, :],
        "abs_PC1_loading": np.abs(vt[0, :]),
        "abs_PC2_loading": np.abs(vt[1, :]),
    }).sort_values("abs_PC1_loading", ascending=False)
    formula_text = pc_formula_text(loading_df, explained_ratio, top_n=3)

    n_clusters = min(3, len(df))
    labels = simple_kmeans(X, n_clusters=n_clusters, random_state=0)
    df["cluster"] = labels
    df["pca_1"] = xy[:, 0]
    df["pca_2"] = xy[:, 1]
    df.attrs["cluster_features"] = feature_cols
    df.attrs["pc_loadings"] = loading_df
    df.attrs["pc_explained_variance_ratio"] = {
        "PC1": float(explained_ratio[0]),
        "PC2": float(explained_ratio[1]),
    }
    df.attrs["pc_formula_text"] = formula_text

    selected_idx = None
    if selected_roi is not None and "roi_0" in df.columns:
        roi_cols = ["roi_0", "roi_1", "roi_2", "roi_3"]
        roi_values = np.array(selected_roi)
        distances = np.linalg.norm(df[roi_cols].to_numpy(dtype=float) - roi_values, axis=1)
        selected_idx = int(np.argmin(distances))

    fig, ax = plt.subplots(figsize=(6.8, 5.8))
    scatter = ax.scatter(df["pca_1"], df["pca_2"], c=df["cluster"], cmap="tab10", s=55, alpha=0.85)
    if selected_idx is not None:
        ax.scatter(
            df.loc[selected_idx, "pca_1"],
            df.loc[selected_idx, "pca_2"],
            s=190,
            marker="*",
            color="red",
            edgecolor="black",
            linewidth=0.8,
            label="selected ROI",
        )
        ax.legend(frameon=False)
    ax.set_xlabel("PCA 1")
    ax.set_ylabel("PCA 2")
    ax.set_title(title)
    fig.colorbar(scatter, ax=ax, label="candidate cluster")
    fig.subplots_adjust(bottom=0.28, right=0.88)
    fig.text(
        0.08,
        0.035,
        formula_text,
        fontsize=7.5,
        va="bottom",
        ha="left",
        bbox={"boxstyle": "round,pad=0.35", "facecolor": "white", "edgecolor": "0.65", "alpha": 0.95},
    )
    if save_path is not None:
        save_path = Path(save_path)
        save_path.parent.mkdir(exist_ok=True, parents=True)
        plt.savefig(save_path, dpi=200, bbox_inches="tight")
        print(f"Saved cluster plot: {save_path}")
    plt.show()
    return df


## 5. 第一步：自动寻找眼睛 ROI

这一轮先在整张图上增强局部暗结构，目标不是直接找 pupil，而是找到包含眼睛和反光点的 ROI。

In [ ]:
def auto_find_eye_roi(gray, debug=True):
    """
    第一轮自动寻找眼睛 ROI。

    现在使用尺度自适应阈值，并把“横向暗椭圆眼裂”作为候选 ROI 的特征之一。
    优先寻找小而圆的暗种子点作为 pupil/iris 候选，再围绕它扩展 ROI；
    如果没有种子候选，再回退到较大的暗区域 ROI 搜索。
    """
    H, W = gray.shape
    params = full_image_scale_params(gray.shape)

    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
    enhanced = clahe.apply(gray)

    low_full = cv2.GaussianBlur(
        enhanced, ksize=(0, 0), sigmaX=params["full_blur_sigma"]
    )

    kernel = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE,
        (params["full_blackhat_kernel"], params["full_blackhat_kernel"]),
    )
    blackhat = cv2.morphologyEx(low_full, cv2.MORPH_BLACKHAT, kernel)

    _, dark_mask = cv2.threshold(
        blackhat, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )
    dark_mask = cv2.morphologyEx(
        dark_mask,
        cv2.MORPH_OPEN,
        cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5)),
    )

    contours, _ = cv2.findContours(
        dark_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )

    seed_candidates = []
    fallback_candidates = []

    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area < params["seed_area_min"] or area > params["fallback_area_max"]:
            continue

        x, y, w, h = cv2.boundingRect(cnt)
        if x <= 2 or y <= 2 or x + w >= W - 2 or y + h >= H - 2:
            continue

        perimeter = cv2.arcLength(cnt, True)
        if perimeter == 0:
            continue

        circularity = 4 * np.pi * area / (perimeter ** 2)
        if circularity < 0.15:
            continue

        M = cv2.moments(cnt)
        if M["m00"] == 0:
            continue
        cx = M["m10"] / M["m00"]
        cy = M["m01"] / M["m00"]

        dark_score = np.mean(blackhat[y:y + h, x:x + w])

        if (
            params["seed_area_min"] <= area <= params["seed_area_max"]
            and circularity >= 0.45
        ):
            roi_w = max(params["seed_min_roi_w"], int(w * 4.0))
            roi_h = max(params["seed_min_roi_h"], int(h * 4.0))
            roi_x = int(cx - roi_w * 0.52)
            roi_y = int(cy - roi_h * 0.52)
            roi_x = max(0, min(roi_x, W - 1))
            roi_y = max(0, min(roi_y, H - 1))
            roi_w = min(roi_w, W - roi_x)
            roi_h = min(roi_h, H - roi_y)

            roi_gray = gray[roi_y:roi_y + roi_h, roi_x:roi_x + roi_w]
            fissure = estimate_eye_fissure_features(roi_gray)
            seed_mask = np.zeros_like(gray)
            cv2.drawContours(seed_mask, [cnt], -1, 255, -1)
            mean_seed_intensity = cv2.mean(gray, mask=seed_mask)[0]
            bright_ratio = np.mean(roi_gray > max(220, np.percentile(gray, 99.3)))

            center_prior = 1 - abs(cx - W * 0.55) / (W * 0.55)
            score = (
                dark_score
                + 25 * circularity
                + 0.01 * area
                - 0.22 * mean_seed_intensity
                + 8 * min(bright_ratio, 0.03)
                + 12 * fissure["fissure_score"]
                + 4 * center_prior
            )

            seed_candidates.append({
                "score": score,
                "contour": cnt,
                "bbox": (x, y, w, h),
                "roi": (roi_x, roi_y, roi_w, roi_h),
                "area": area,
                "circularity": circularity,
                "dark_score": dark_score,
                "bright_ratio": bright_ratio,
                "mean_seed_intensity": mean_seed_intensity,
                "mode": "dark circular seed",
                **fissure,
            })

        if params["fallback_area_min"] <= area <= params["fallback_area_max"]:
            roi_x, roi_y, roi_w, roi_h = expand_bbox(
                x, y, w, h, gray.shape, scale_x=2.5, scale_y=2.2
            )
            roi_gray = gray[roi_y:roi_y + roi_h, roi_x:roi_x + roi_w]
            fissure = estimate_eye_fissure_features(roi_gray)
            bright_threshold = max(220, np.percentile(gray, 99.3))
            bright_pixels = np.sum(roi_gray > bright_threshold)
            bright_ratio = bright_pixels / max(1, roi_gray.size)
            score = (
                dark_score
                + 60 * min(bright_ratio, 0.02)
                + 0.002 * area
                + 15 * circularity
                + 12 * fissure["fissure_score"]
            )

            fallback_candidates.append({
                "score": score,
                "contour": cnt,
                "bbox": (x, y, w, h),
                "roi": (roi_x, roi_y, roi_w, roi_h),
                "area": area,
                "circularity": circularity,
                "dark_score": dark_score,
                "bright_ratio": bright_ratio,
                "mode": "fallback dark region",
                **fissure,
            })

    candidates = seed_candidates if len(seed_candidates) > 0 else fallback_candidates
    if len(candidates) == 0:
        raise RuntimeError("没有自动找到眼睛 ROI。可以放宽 area / circularity 参数，或手动给 ROI。")

    best = max(candidates, key=lambda d: d["score"])
    roi = best["roi"]

    if debug:
        show_img("CLAHE enhanced full image", enhanced)
        show_img("First low-pass filtered full image", low_full)
        show_img("Black-hat response for eye ROI search", blackhat)
        show_img("Dark mask from first round", dark_mask)

        overlay = np.zeros((*gray.shape, 4), dtype=np.float32)
        x, y, w, h = best["bbox"]
        rx, ry, rw, rh = roi
        cv2.rectangle(overlay, (x, y), (x + w, y + h), (1, 0, 0, 1), 2)
        cv2.rectangle(overlay, (rx, ry), (rx + rw, ry + rh), (0, 1, 0, 1), 2)
        show_overlay("Auto detected eye ROI: red = candidate, green = final ROI", gray, overlay)

        print("Scale-adaptive full-image parameters:")
        display(pd.DataFrame([params]))
        display(pd.DataFrame([{k: round(v, 5) if isinstance(v, float) else v
                               for k, v in best.items() if k not in {"contour"}}]))

    return roi, candidates


In [ ]:
roi, roi_candidates = auto_find_eye_roi(gray, debug=debug)
print("Auto ROI:", roi)

## 5.1 候选 ROI 聚类图

使用候选特征做标准化、PCA 和 KMeans 聚类。红色星号是最终选中的 ROI 候选。

In [ ]:
cluster_plot_path = Path("pupil_output") / f"{img_path.stem}_roi_candidate_clusters.png"
roi_candidate_df = plot_candidate_clusters(
    roi_candidates,
    selected_roi=roi,
    title="ROI candidate feature clusters",
    save_path=cluster_plot_path,
)

print("Cluster features:")
print(", ".join(roi_candidate_df.attrs["cluster_features"]))
print("PC explained variance ratio:", roi_candidate_df.attrs["pc_explained_variance_ratio"])
print("PC formula text:")
print(roi_candidate_df.attrs["pc_formula_text"])

print("PC loadings. Larger absolute values contribute more to that PC axis:")
display(roi_candidate_df.attrs["pc_loadings"].round(4))

display(roi_candidate_df.sort_values("score", ascending=False).head(10).round(4))


## 6. 第二步：在 ROI 内提取 pupil

这一轮只处理 ROI：先去除高亮反光点，再用 black-hat 增强暗圆形结构，最后筛选轮廓并拟合圆。

In [ ]:
def compute_pupil_qc(result, roi_shape):
    """Compute automatic QC metrics and a heuristic confidence score for a fitted pupil."""
    h, w = roi_shape
    cand = result["candidate"]
    contour = result["contour_roi"]
    cx_roi, cy_roi = result["center_roi"]
    r = result["radius_px"]
    residual = circle_fit_residual(contour, cx_roi, cy_roi, r)
    edge_margin = min(cx_roi, cy_roi, w - cx_roi, h - cy_roi)
    score_values = sorted([c["score"] for c in result["candidates"]])
    score_margin = score_values[1] - score_values[0] if len(score_values) > 1 else np.nan
    area = cand["area"]
    circle_area = np.pi * r * r
    area_ratio_to_circle = area / circle_area if circle_area > 0 else np.nan

    warnings = []
    if cand["circularity"] < 0.5:
        warnings.append("low_circularity")
    if residual > max(1.5, 0.18 * r):
        warnings.append("high_circle_residual")
    if edge_margin < max(3, 0.25 * r):
        warnings.append("pupil_near_roi_edge")
    if not np.isnan(score_margin) and score_margin < 3:
        warnings.append("ambiguous_candidate_score")
    if area_ratio_to_circle < 0.35 or area_ratio_to_circle > 1.35:
        warnings.append("contour_circle_area_mismatch")

    residual_ref = max(1.5, 0.18 * r)
    circularity_score = float(np.clip((cand["circularity"] - 0.35) / 0.55, 0, 1))
    residual_score = float(np.clip(1 - residual / (2 * residual_ref), 0, 1))
    edge_score = float(np.clip(edge_margin / max(3, 1.5 * r), 0, 1))
    margin_score = 0.65 if np.isnan(score_margin) else float(np.clip(score_margin / 10, 0, 1))
    area_score = float(np.clip(1 - abs(area_ratio_to_circle - 1) / 0.65, 0, 1))

    confidence_raw = (
        0.25 * circularity_score
        + 0.25 * residual_score
        + 0.20 * edge_score
        + 0.15 * margin_score
        + 0.15 * area_score
    )
    confidence = float(np.clip(confidence_raw - 0.12 * len(warnings), 0, 1))
    if confidence >= 0.85:
        confidence_level = "high"
    elif confidence >= 0.65:
        confidence_level = "medium"
    else:
        confidence_level = "low"

    return {
        "qc_pass": len(warnings) == 0,
        "qc_confidence": confidence,
        "qc_confidence_level": confidence_level,
        "qc_warnings": ";".join(warnings),
        "qc_circularity": cand["circularity"],
        "qc_contour_area": area,
        "qc_circle_residual_px": residual,
        "qc_edge_margin_px": float(edge_margin),
        "qc_score_margin": float(score_margin) if not np.isnan(score_margin) else np.nan,
        "qc_area_ratio_to_circle": float(area_ratio_to_circle),
        "qc_num_candidates": len(result["candidates"]),
        "qc_circularity_score": circularity_score,
        "qc_residual_score": residual_score,
        "qc_edge_score": edge_score,
        "qc_margin_score": margin_score,
        "qc_area_score": area_score,
    }


def extract_pupil_from_roi(gray, roi, debug=True):
    x, y, w, h = roi
    roi_raw = gray[y:y + h, x:x + w].copy()
    params = roi_scale_params(roi_raw.shape)

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(6, 6))
    roi_enhanced = clahe.apply(roi_raw)

    glint_threshold = max(220, np.percentile(roi_raw, 99.3))
    params["glint_threshold"] = glint_threshold
    glint_mask = (roi_raw > glint_threshold).astype(np.uint8) * 255
    glint_mask = cv2.dilate(
        glint_mask,
        cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5)),
        iterations=1,
    )

    roi_no_glint = cv2.inpaint(
        roi_enhanced, glint_mask, inpaintRadius=3, flags=cv2.INPAINT_TELEA
    )

    low_roi = cv2.GaussianBlur(
        roi_no_glint, ksize=(0, 0), sigmaX=params["roi_blur_sigma"]
    )

    pupil_kernel = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE,
        (params["pupil_blackhat_kernel"], params["pupil_blackhat_kernel"]),
    )
    blackhat = cv2.morphologyEx(low_roi, cv2.MORPH_BLACKHAT, pupil_kernel)

    _, binary = cv2.threshold(
        blackhat, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )
    binary = cv2.morphologyEx(
        binary,
        cv2.MORPH_OPEN,
        cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3)),
    )
    binary = cv2.morphologyEx(
        binary,
        cv2.MORPH_CLOSE,
        cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5)),
    )

    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    candidates = []
    roi_center = np.array([w * 0.45, h * 0.45])

    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area < params["pupil_area_min"] or area > params["pupil_area_max"]:
            continue

        perimeter = cv2.arcLength(cnt, True)
        if perimeter == 0:
            continue

        circularity = 4 * np.pi * area / (perimeter ** 2)
        if circularity < 0.35:
            continue

        M = cv2.moments(cnt)
        if M["m00"] == 0:
            continue

        cx = M["m10"] / M["m00"]
        cy = M["m01"] / M["m00"]

        mask = np.zeros_like(roi_raw)
        cv2.drawContours(mask, [cnt], -1, 255, -1)
        mean_intensity = cv2.mean(low_roi, mask=mask)[0]
        dist_to_center = np.linalg.norm(np.array([cx, cy]) - roi_center)

        score = mean_intensity - 35 * circularity + 0.25 * dist_to_center + 0.01 * area
        candidates.append({
            "score": score,
            "contour": cnt,
            "area": area,
            "circularity": circularity,
            "center": (cx, cy),
            "mean_intensity": mean_intensity,
        })

    if len(candidates) == 0:
        raise RuntimeError("没有找到 pupil 候选轮廓。可以调低面积阈值，或调整 black-hat kernel。")

    best = min(candidates, key=lambda d: d["score"])
    pupil_cnt_roi = best["contour"]

    cx_roi, cy_roi, r = fit_circle_least_squares(pupil_cnt_roi)
    diameter_px = 2 * r
    cx_global = cx_roi + x
    cy_global = cy_roi + y

    result = cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)
    cv2.rectangle(result, (x, y), (x + w, y + h), (255, 0, 0), 2)
    pupil_cnt_global = pupil_cnt_roi + np.array([[[x, y]]])
    cv2.drawContours(result, [pupil_cnt_global], -1, (0, 255, 0), 2)
    cv2.circle(result, (int(round(cx_global)), int(round(cy_global))), int(round(r)), (255, 0, 0), 2)
    cv2.circle(result, (int(round(cx_global)), int(round(cy_global))), 2, (255, 255, 0), -1)

    output = {
        "roi": roi,
        "center_roi": (cx_roi, cy_roi),
        "center_global": (cx_global, cy_global),
        "radius_px": r,
        "diameter_px": diameter_px,
        "contour_roi": pupil_cnt_roi,
        "result_image": result,
        "binary_mask": binary,
        "low_roi": low_roi,
        "candidate": best,
        "candidates": candidates,
        "scale_params": params,
    }
    output["qc"] = compute_pupil_qc(output, roi_raw.shape)
    label = f"diam={diameter_px:.1f}px  QC={output['qc']['qc_confidence']:.2f} ({output['qc']['qc_confidence_level']})"
    cv2.putText(
        result,
        label,
        (12, 28),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.65,
        (255, 255, 0),
        2,
        cv2.LINE_AA,
    )

    if debug:
        show_img("ROI raw image", roi_raw)
        show_img("ROI CLAHE enhanced", roi_enhanced)
        show_img("Glint mask", glint_mask)
        show_img("ROI after glint inpainting", roi_no_glint)
        show_img("Second low-pass filtered ROI", low_roi)
        show_img("ROI black-hat response for pupil", blackhat)
        show_img("Pupil binary mask", binary)

        contour_vis = cv2.cvtColor(roi_raw, cv2.COLOR_GRAY2RGB)
        cv2.drawContours(contour_vis, [pupil_cnt_roi], -1, (0, 255, 0), 2)
        cv2.circle(contour_vis, (int(round(cx_roi)), int(round(cy_roi))), int(round(r)), (255, 0, 0), 2)
        show_img("Pupil contour and fitted circle inside ROI", contour_vis, cmap=None)
        show_img("Final pupil boundary and fitted circle", result, cmap=None, figsize=(8, 6))

        print("Scale-adaptive ROI parameters:")
        display(pd.DataFrame([params]))
        display(pd.DataFrame([{k: round(v, 5) if isinstance(v, float) else v
                               for k, v in best.items() if k != "contour"}]))
        print("Automatic QC metrics:")
        display(pd.DataFrame([output["qc"]]))

    return output


In [ ]:
result = extract_pupil_from_roi(gray, roi, debug=debug)

## 7. 汇总 pupil 直径结果

In [ ]:
summary = {
    "image": str(img_path),
    "roi_x": result["roi"][0],
    "roi_y": result["roi"][1],
    "roi_w": result["roi"][2],
    "roi_h": result["roi"][3],
    "center_x_px": result["center_global"][0],
    "center_y_px": result["center_global"][1],
    "radius_px": result["radius_px"],
    "diameter_px": result["diameter_px"],
    **result["qc"],
}

if micron_per_px is not None:
    summary["diameter_um"] = result["diameter_px"] * micron_per_px

summary_df = pd.DataFrame([summary])
display(summary_df.round(3))


## 8. 保存可视化结果和 CSV

In [ ]:
out_dir = Path("pupil_output")
out_dir.mkdir(exist_ok=True)

result_img_path = out_dir / f"{img_path.stem}_pupil_result.png"
summary_csv_path = out_dir / f"{img_path.stem}_pupil_summary.csv"

cv2.imwrite(str(result_img_path), cv2.cvtColor(result["result_image"], cv2.COLOR_RGB2BGR))
summary_df.to_csv(summary_csv_path, index=False)

print(f"Saved result image: {result_img_path}")
print(f"Saved summary CSV: {summary_csv_path}")

## 9. 可选：手动指定 ROI

如果自动 ROI 在某张图片上失败，可以跳过第 5 步，手动填入 `(x, y, w, h)` 后重新运行 pupil 提取。

In [ ]:
# manual_roi = (120, 80, 160, 120)
# result_manual = extract_pupil_from_roi(gray, manual_roi, debug=True)
# print("Manual ROI pupil diameter:", round(result_manual["diameter_px"], 2), "px")

## 10. 视频分析：单视频与批量处理

下面的 cell 使用已经打包好的 `pupil_finding` Python 模块进行视频分析。输出会自动按 session 保存，并生成逐帧 CSV、时间-pupil 直径图、相对变化率图和可选标注视频。


### 10.1 导入视频分析工具


In [ ]:
from pathlib import Path
from IPython.display import Image as IPyImage, Video as IPyVideo
from pupil_finding.video import make_session_dir, process_video, process_video_batch


### 10.2 单个视频分析

把 `video_path` 改成要分析的视频路径。`analysis_fps` 用于把 frame index 转换为秒；如果保持 `None`，则使用视频文件元数据中的 FPS。


In [ ]:
video_path = Path("path/to/video.mp4")
video_output_root = Path("pupil_video_output")
video_session_name = "demo_video_session"

analysis_fps = 30       # 例如 30；如果想用视频自身 FPS，改成 None
every_n = 1             # 每 N 帧分析一次
start_frame = 0
max_frames = None       # 调试时可设为 100；完整分析保持 None
redetect_every = 30     # 每分析 N 帧重新自动找 ROI
micron_per_px = None    # 如果有标尺，例如 3.2 um/px，就填 3.2
write_annotated_video = True


In [ ]:
if not video_path.exists():
    print(f"Video not found: {video_path}")
    print("请先修改 video_path，然后重新运行本 cell。")
else:
    video_session_dir = make_session_dir(video_output_root, video_session_name)
    single_video_output_dir = video_session_dir / video_path.stem

    video_df = process_video(
        video_path,
        single_video_output_dir,
        every_n=every_n,
        start_frame=start_frame,
        max_frames=max_frames,
        redetect_every=redetect_every,
        micron_per_px=micron_per_px,
        write_annotated_video=write_annotated_video,
        analysis_fps=analysis_fps,
    )

    print(f"Saved outputs to: {single_video_output_dir}")
    display(video_df.head())
    display(video_df.describe(include="all"))


### 10.3 查看单视频输出图像和标注视频


In [ ]:
if video_path.exists():
    diameter_plot = single_video_output_dir / f"{video_path.stem}_diameter_timeseries.png"
    relative_plot = single_video_output_dir / f"{video_path.stem}_diameter_relative_change.png"
    annotated_video = single_video_output_dir / f"{video_path.stem}_annotated.mp4"
    csv_path = single_video_output_dir / f"{video_path.stem}_pupil_timeseries.csv"

    print(f"CSV: {csv_path}")
    if diameter_plot.exists():
        display(IPyImage(filename=str(diameter_plot)))
    if relative_plot.exists():
        display(IPyImage(filename=str(relative_plot)))
    if annotated_video.exists():
        display(IPyVideo(str(annotated_video), embed=False))
else:
    print("video_path 不存在；先完成 10.2 的路径设置和分析。")


### 10.4 批量视频分析

`batch_inputs` 可以是视频文件、文件夹或 glob pattern。批量输出会保存在同一个 session 文件夹下，每个视频一个子文件夹，并额外生成 `batch_summary.csv`。


In [ ]:
batch_inputs = [Path("path/to/video_folder")]
batch_output_root = Path("pupil_video_output")
batch_session_name = "demo_batch_session"

batch_recursive = False
batch_analysis_fps = 30
batch_every_n = 1
batch_start_frame = 0
batch_max_frames = None
batch_redetect_every = 30
batch_micron_per_px = None
batch_write_annotated_video = True


In [ ]:
valid_batch_inputs = []
for item in batch_inputs:
    if any(ch in str(item) for ch in "*?[]") or Path(item).exists():
        valid_batch_inputs.append(item)

if not valid_batch_inputs:
    print("没有找到有效的 batch_inputs。请修改为视频文件、视频文件夹或 glob pattern。")
else:
    batch_session_dir, batch_summary = process_video_batch(
        valid_batch_inputs,
        batch_output_root,
        session_name=batch_session_name,
        recursive=batch_recursive,
        every_n=batch_every_n,
        start_frame=batch_start_frame,
        max_frames=batch_max_frames,
        redetect_every=batch_redetect_every,
        micron_per_px=batch_micron_per_px,
        write_annotated_video=batch_write_annotated_video,
        analysis_fps=batch_analysis_fps,
    )

    print(f"Saved batch session to: {batch_session_dir}")
    display(batch_summary)


### 10.5 读取已有 session 的结果

如果已经分析过视频，可以直接读取 session 下的 CSV 文件重新作图或统计。


In [ ]:
existing_session_dir = Path("pupil_video_output/demo_video_session")

if existing_session_dir.exists():
    csv_files = sorted(existing_session_dir.glob("*/*_pupil_timeseries.csv"))
    print(f"Found {len(csv_files)} CSV files")
    for csv_file in csv_files[:5]:
        print(csv_file)
else:
    print(f"Session folder not found: {existing_session_dir}")
